# NSCLC Clinical Data Quality Exploration

This portfolio project profiles sample-level clinical data, checks completeness and consistency, and produces a simulated review-query log for a Power BI workflow.

**Source:** AACR Project GENIE BPC NSCLC v2.0-public, via [cBioPortal](https://genie.cbioportal.org/study/summary?id=nsclc_public_genie_bpc). See the [dataset description](https://aacrprojectgenie.org/data/bpc-nsclc-v2-0-public/) and [usage and attribution guidance](https://aacrprojectgenie.org/faq/).

The original notebook reported 2,004 samples from 1,846 patients. These historical counts are context, not hardcoded validation targets. Multiple samples from the same patient are not automatically duplicate records.

**Interpretation:** flags identify potential review items, not confirmed clinical errors. Query resolution is simulated; source-document verification has not been performed. These are exploratory rules and need checking against the source data dictionary.

This revised copy removes patient-level displayed outputs and makes query generation repeatable. The user-run saved notebook reports 7 flagged sample records across 6 patients (0.35%) and 7 Open queries. The other five rules have no flags. This public copy has saved outputs cleared; rerun using your authorized source export.

## 1. Setup and source file

Use Python with pandas and NumPy. Put your authorized clinical TSV export in `data/`, relative to the repository root. Alternatively, set `NSCLC_DATA_PATH` to its location. Source records are not included in this project. The original file is read only.

If using an iCloud-backed file, download it in Finder before running this notebook.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

DATA_PATH = Path(os.environ.get(
    "NSCLC_DATA_PATH",
    str(PROJECT_ROOT / "data" / "nsclc_public_genie_bpc_clinical_data.tsv")
)).expanduser()

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}"
    )

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    dtype={"Patient ID": "string", "Sample ID": "string"}
)

print(f"Loaded {len(df):,} sample records and {len(df.columns):,} columns.")
print(f"Python dependencies: pandas {pd.__version__}; NumPy {np.__version__}")


## 2. Unit of analysis and duplicate checks

Count samples and patients separately. This notebook preserves every source row and does not silently deduplicate. Missing identifiers or duplicate sample IDs stop validation because queries require a unique sample key.

In [ ]:
required_ids = ["Patient ID", "Sample ID"]
missing_ids = [column for column in required_ids if column not in df.columns]
if missing_ids:
    raise ValueError(f"Missing required identifier columns: {missing_ids}")

id_values = df[required_ids].apply(lambda s: s.str.strip().replace("", pd.NA))
samples_per_patient = id_values.groupby("Patient ID")["Sample ID"].nunique()
profile = pd.Series({
    "Sample records": len(df),
    "Unique patients": id_values["Patient ID"].nunique(),
    "Unique samples": id_values["Sample ID"].nunique(),
    "Source columns": len(df.columns),
    "Patients with multiple samples": int((samples_per_patient > 1).sum()),
    "Maximum samples per patient": samples_per_patient.max(),
    "Completely duplicated rows": int(df.duplicated().sum()),
    "Duplicate sample IDs": int(id_values["Sample ID"].duplicated().sum()),
    "Duplicate patient-sample pairs": int(id_values.duplicated().sum()),
    "Missing patient IDs": int(id_values["Patient ID"].isna().sum()),
    "Missing sample IDs": int(id_values["Sample ID"].isna().sum()),
}, name="Value")
display(profile.to_frame())
if id_values.isna().any().any() or id_values["Sample ID"].duplicated().any():
    raise ValueError("Resolve missing identifiers or duplicate sample IDs before generating queries.")


## 3. Select clinical fields and profile missingness

Include every source column beginning with `Distant Mets:`. This includes subcutaneous tissue when present, which was omitted from the original eight-site list. The site count refers only to these source fields, not every possible clinical metastatic site.

Whitespace-only text is treated as missing in the working copy. Unknown or not-mentioned categories remain distinct from missing values. No clinical values are corrected or imputed.

In [ ]:
MET_STATUS = "Sites of Distant Metastasis at the Time of Cancer Diagnosis (Stage IV Patients)"
TEST_COL = "PD-L1 Testing at Time of Sample Acquisition"
RESULT_COL = "Positive PD-L1 Result at the Time of Sample Acquisition"
met_columns = [column for column in df.columns if column.startswith("Distant Mets:")]
if not met_columns:
    raise ValueError("No Distant Mets: columns were found; site consistency cannot be evaluated.")

base_columns = [
    "Patient ID", "Sample ID", "Age at sequencing", "Sex", "Histology",
    "Histology Category", "Stage at Diagnosis",
    "Non-Small Cell Lung Cancer: Cigarette Use at Time of Diagnosis",
    "Sample Type", "Number of Samples Per Patient", MET_STATUS,
    "Number of imaging reports", "Number of medical oncologist notes curated",
    "Number of pathology reports", "Number of cancer-directed drug regimens curated",
    TEST_COL, RESULT_COL,
]
selected_columns = base_columns + met_columns
missing_columns = [column for column in selected_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Source schema differs; required columns are missing: {missing_columns}")

clinical_df = df[selected_columns].copy()
for column in clinical_df.select_dtypes(include=["object", "string"]).columns:
    clinical_df[column] = clinical_df[column].astype("string").str.strip().replace("", pd.NA)

print(f"Working fields: {len(selected_columns)}; metastasis-site fields: {len(met_columns)}")
print("Included site fields:", ", ".join(met_columns))
quality_report = pd.DataFrame({
    "Data type": clinical_df.dtypes.astype(str),
    "Non-missing": clinical_df.notna().sum(),
    "Missing": clinical_df.isna().sum(),
    "Missing percent": (clinical_df.isna().mean() * 100).round(2),
    "Unique values": clinical_df.nunique(),
}).sort_values("Missing percent", ascending=False)
display(quality_report)


## 4. Review category coverage

These summaries show sample-record frequencies, not patient prevalence. Missingness alone is not necessarily an error: some fields apply only to certain stages or testing statuses. PD-L1 rules below cover explicit `Yes` and `No` testing values; missing and other statuses are reported separately.

In [ ]:
for column in ["Sex", "Stage at Diagnosis", "Histology Category", "Sample Type", TEST_COL, RESULT_COL]:
    print(f"\n{column}")
    display(clinical_df[column].value_counts(dropna=False).rename("Sample records").to_frame())

print("PD-L1 testing status missing:", int(clinical_df[TEST_COL].isna().sum()))
print("PD-L1 testing status outside Yes/No (nonmissing):", int(
    (clinical_df[TEST_COL].notna() & ~clinical_df[TEST_COL].isin(["Yes", "No"])).sum()
))


## 5. Apply exploratory QC rules

All flags are at the sample-record level. Stage-dependent checks reflect the fields' stated diagnosis context, not a universal staging rule. Site values other than Yes/No are treated as unresolved for site-completeness checks. An unrecorded site count is left missing rather than interpreted as zero.

The metastasis review flag means the overall field says Yes but **none of the included source site fields** says Yes. It does not prove a clinical contradiction.

In [ ]:
RULES = [
    ("FLAG_Missing_Stage", "Missing Stage at Diagnosis", "Stage at Diagnosis", "Missing Data",
     "Verify missing stage against source documentation."),
    ("FLAG_Missing_Sample_Type", "Missing Sample Type", "Sample Type", "Missing Data",
     "Verify missing sample type against source documentation."),
    ("FLAG_PDL1_Test_No_Result", "PD-L1 Test With Missing Result", RESULT_COL, "Clinical Consistency",
     "Testing is recorded as Yes but the result is missing; verify source documentation."),
    ("FLAG_PDL1_Result_Without_Test", "PD-L1 Result With Testing Recorded No", TEST_COL, "Clinical Consistency",
     "Testing is recorded as No but a result is populated; verify source documentation."),
    ("FLAG_StageIV_Missing_Met_Data", "Stage IV With Missing or Unresolved Site Fields", "Distant Metastasis Site", "Missing or Unresolved Data",
     "Stage IV has site fields missing or outside Yes/No; review applicability and source documentation."),
    ("FLAG_NonStageIV_Has_Met_Data", "Non-Stage-IV With Metastasis Data", "Distant Metastasis Site", "Clinical Consistency",
     "Site data are populated for a known non-Stage-IV diagnosis; verify field scope and source documentation."),
    ("FLAG_Metastasis_Site_Discrepancy", "Overall Metastasis Yes Without a Positive Included Site", "Distant Metastasis Site", "Clinical Consistency",
     "Stage IV and overall metastasis are recorded as Yes, but no included source site is Yes; review full source documentation."),
]

def apply_qc(frame, site_columns):
    if not site_columns:
        raise ValueError("A nonempty site list is required.")
    out = frame.copy()
    stage4 = out["Stage at Diagnosis"].eq("Stage IV").fillna(False)
    non_stage4 = out["Stage at Diagnosis"].notna() & ~stage4
    sites = out[site_columns]
    yes_counts = sites.eq("Yes").fillna(False).sum(axis=1)
    any_known_site = sites.isin(["Yes", "No"]).any(axis=1)
    out["Number_of_Metastatic_Sites"] = yes_counts.astype("Int64").where(any_known_site)
    out["FLAG_Missing_Stage"] = out["Stage at Diagnosis"].isna()
    out["FLAG_Missing_Sample_Type"] = out["Sample Type"].isna()
    out["FLAG_PDL1_Test_No_Result"] = out[TEST_COL].eq("Yes").fillna(False) & out[RESULT_COL].isna()
    out["FLAG_PDL1_Result_Without_Test"] = out[TEST_COL].eq("No").fillna(False) & out[RESULT_COL].notna()
    out["FLAG_StageIV_Missing_Met_Data"] = stage4 & ~sites.isin(["Yes", "No"]).all(axis=1)
    out["FLAG_NonStageIV_Has_Met_Data"] = non_stage4 & sites.notna().any(axis=1)
    out["FLAG_Metastasis_Site_Discrepancy"] = (
        stage4 & out[MET_STATUS].eq("Yes").fillna(False) & yes_counts.eq(0)
    )
    flags = [rule[0] for rule in RULES]
    out["Any_QC_Flag"] = out[flags].any(axis=1)
    return out

validation_df = apply_qc(clinical_df, met_columns)
flag_columns = [rule[0] for rule in RULES]
qc_summary = pd.DataFrame({
    "Validation Rule": [rule[1] for rule in RULES],
    "Records Flagged": [int(validation_df[rule[0]].sum()) for rule in RULES],
})
qc_summary["Percent of Records"] = (
    (qc_summary["Records Flagged"] / len(validation_df) * 100).round(2)
    if len(validation_df) else np.nan
)
qc_summary["QC Status"] = np.where(qc_summary["Records Flagged"].eq(0), "No flags under this rule", "Review Required")
display(qc_summary)
flagged_records = int(validation_df["Any_QC_Flag"].sum())
print("Sample records requiring review:", flagged_records)
print("Unique patients with one or more flags:", validation_df.loc[validation_df["Any_QC_Flag"], "Patient ID"].nunique())
print("Sample records with no current flags:", len(validation_df) - flagged_records)
if len(validation_df):
    print(f"Sample-record review rate: {flagged_records / len(validation_df):.2%}")


## 6. Repeatable simulated query log

One query is created for each flagged sample–rule pair. A sample can have more than one query, so query count and flagged-record count need not match. The function creates a fresh list on every call and covers all seven rules, including missing sample type. The displayed summary contains no patient identifiers.

All queries start Open. Query IDs are sequential within a run and should not be used as persistent cross-version identifiers. The sample ID and rule together identify the review item.

In [ ]:
def build_query_log(validated):
    queries = []
    for flag, rule_name, field, issue_type, query_text in RULES:
        flagged = validated.loc[validated[flag], ["Patient ID", "Sample ID"]].sort_values(["Patient ID", "Sample ID"])
        for patient_id, sample_id in flagged.itertuples(index=False, name=None):
            queries.append({
                "Patient_ID": patient_id, "Sample_ID": sample_id, "Rule": flag,
                "Field": field, "Issue_Type": issue_type,
                "Query_Text": query_text, "Status": "Open",
            })
    result = pd.DataFrame(queries, columns=[
        "Patient_ID", "Sample_ID", "Rule", "Field", "Issue_Type", "Query_Text", "Status"
    ])
    result.insert(0, "Query_ID", [f"QRY-{i:04d}" for i in range(1, len(result) + 1)])
    if result.duplicated(["Sample_ID", "Rule"]).any():
        raise ValueError("Duplicate sample-rule queries detected; inspect the input records.")
    return result

query_log = build_query_log(validation_df)
expected_queries = int(validation_df[flag_columns].sum().sum())
assert len(query_log) == expected_queries, "Query count must match sample-rule flags."
pd.testing.assert_frame_equal(query_log, build_query_log(validation_df))
assert query_log["Sample_ID"].nunique() == flagged_records
print("Queries generated:", len(query_log))
print("All query statuses are Open:", bool(query_log["Status"].eq("Open").all()))
display(query_log.groupby(["Field", "Issue_Type", "Status"]).size().rename("Queries").reset_index())


## 7. Optional local exports

Exports are disabled by default. If needed for Power BI, set `EXPORT_LOCAL_FILES = True`. Patient-level validation data and queries go only into `local_exports/`, excluded by the supplied `.gitignore`. They are local working files, not approved public deliverables. “QC annotated” means flags were added; it does not mean source discrepancies were resolved.

Review aggregate outputs and source usage terms before sharing them. A `.gitignore` prevents accidental additions of untracked files; it does not remove files previously committed to Git.

In [ ]:
EXPORT_LOCAL_FILES = False
if EXPORT_LOCAL_FILES:
    export_dir = PROJECT_ROOT / "local_exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    validation_df.to_csv(export_dir / "qc_annotated_clinical_data.csv", index=False)
    query_log.to_csv(export_dir / "clinical_query_log.csv", index=False)
    qc_summary.to_csv(export_dir / "clinical_qc_summary.csv", index=False)
    quality_report.to_csv(export_dir / "clinical_missingness_summary.csv")
    print("Created local exports. Do not commit patient-level files to the public repository.")
else:
    print("Local exports disabled; no source or patient-level output files were written.")


## 8. Power BI interpretation and next steps

- Use distinct nonblank patient IDs for patient totals; use sample IDs for sample totals.
- Label age as **age at sequencing**. For patient-level age summaries, define which sequencing event represents each patient before grouping or taking a median. This notebook does not silently choose one.
- Include or explain unknown and missing histology/stage categories. Check whether patient attributes vary across samples before assuming category counts sum to the patient total.
- Patients may have more than one metastatic site. Site counts should not be summed to estimate unique patients.
- Keep open review queries distinct from confirmed errors and resolved corrections.
- Check the revised results against the prior exports; additional site coverage can change flags and counts. Do not carry forward the old count of seven metastasis flags without recomputing.

### Attribution

The AACR Project GENIE Consortium. *AACR Project GENIE: Powering Precision Medicine Through An International Consortium.* Cancer Discovery. 2017;7(8):818–831. Dataset: BPC NSCLC v2.0-public.

See also [The GENIE BPC NSCLC Cohort](https://aacrjournals.org/clincancerres/article/29/17/3418/728542/The-GENIE-BPC-NSCLC-Cohort-A-Real-World-Repository), Clinical Cancer Research (2023).

Before publication, add the actual data-access date and the required acknowledgment from the official AACR attribution guidance. Interpretations in this independent portfolio project are the author's responsibility.